## Config

In [1]:
import os
import csv
import math
import random
from dataclasses import dataclass
from typing import Dict, List, Tuple

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

from torch.amp import GradScaler, autocast

DATA_DIR = 'data'
VOCAB_PATH = os.path.join(DATA_DIR, 'vocab.txt')
META_CSV = os.path.join(DATA_DIR, 'metadata_clean.csv')
TOK_DIR = os.path.join(DATA_DIR, 'tokenized')

# ---------- Training config ----------
SEED = 42
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

SEQ_LEN = 50        # input length
STRIDE = 10         # step between windows (reduce redundancy)
BATCH_SIZE = 256
EPOCHS = 8
LR = 1e-3
GRAD_CLIP = 1.0

EMB_DIM = 256
HIDDEN_SIZE = 512
NUM_LAYERS = 2
DROPOUT = 0.2

# Per-book split ratios (each book is split into 3 contiguous parts)
TRAIN_RATIO = 0.8
VAL_RATIO   = 0.1
TEST_RATIO  = 0.1

MIN_BOOK_TOKENS = 2000  # skip tiny books

STEPS_PER_EPOCH = 3000
CKPT_DIR = 'checkpoints'
SAVE_EVERY_EPOCH = True
SAVE_EVERY_STEPS = 200
# ------------------------------------

use_amp = (DEVICE == "cuda")

print('DEVICE:', DEVICE)
os.makedirs(CKPT_DIR, exist_ok=True)

def set_seed(seed: int):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

DEVICE: cuda


## Load vocab + metadata

In [2]:
def load_vocab(vocab_path: str) -> Tuple[List[str], Dict[str, int]]:
    if not os.path.exists(vocab_path):
        raise FileNotFoundError(f'vocab.txt not found: {vocab_path}')
    with open(vocab_path, 'r', encoding='utf-8') as f:
        vocab = [line.strip() for line in f if line.strip()]
    tok2id = {t: i for i, t in enumerate(vocab)}
    return vocab, tok2id

def read_metadata(meta_csv: str) -> List[dict]:
    if not os.path.exists(meta_csv):
        raise FileNotFoundError(f'metadata.csv not found: {meta_csv}')
    with open(meta_csv, 'r', encoding='utf-8', newline='') as f:
        return list(csv.DictReader(f))

vocab, tok2id = load_vocab(VOCAB_PATH)
rows = read_metadata(META_CSV)

pad_id = tok2id.get('<pad>', 0)
bos_id = tok2id.get('<bos>')
eos_id = tok2id.get('<eos>')

print('vocab size:', len(vocab))
print('special ids:', {'<pad>': pad_id, '<bos>': bos_id, '<eos>': eos_id})
print('metadata rows:', len(rows))

vocab size: 26286
special ids: {'<pad>': 0, '<bos>': 2, '<eos>': 3}
metadata rows: 49


## Dataset (windowed next-token prediction)

In [3]:
def load_ids_file(path: str) -> List[int]:
    with open(path, "r", encoding="utf-8") as f:
        s = f.read().strip()
    return list(map(int, s.split())) if s else []

@dataclass
class BookData:
    ebook_id: str
    ids_path: str
    token_count: int

def collect_books(rows: List[dict]) -> List[BookData]:
    books = []
    for r in rows:
        ebook_id = r.get('ebook_id')
        if not ebook_id:
            continue
        ids_path = os.path.join(TOK_DIR, f'{ebook_id}.ids.txt')
        if not os.path.exists(ids_path):
            continue
        try:
            token_count = int(r.get('token_count', '0'))
        except Exception:
            token_count = 0
        if token_count < MIN_BOOK_TOKENS:
            continue
        books.append(BookData(str(ebook_id), ids_path, token_count))
    return books

def split_book_tokens(ids: List[int], train_r: float, val_r: float) -> Tuple[List[int], List[int], List[int]]:
    """Split a single book's tokens into train/val/test contiguous segments."""
    n = len(ids)
    train_end = int(n * train_r)
    val_end = int(n * (train_r + val_r))
    return ids[:train_end], ids[train_end:val_end], ids[val_end:]

class CachedWindowDataset(Dataset):
    def __init__(self, token_chunks: List[torch.Tensor], seq_len: int, stride: int):
        self.seq_len = seq_len
        self.stride = stride
        self.book_tokens = []
        self.samples: List[Tuple[int, int]] = []

        for chunk in token_chunks:
            if chunk.numel() < seq_len + 2:
                continue
            bi = len(self.book_tokens)
            self.book_tokens.append(chunk)
            max_start = chunk.numel() - (seq_len + 1)
            for st in range(0, max_start + 1, stride):
                self.samples.append((bi, st))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx: int):
        bi, st = self.samples[idx]
        ids = self.book_tokens[bi]
        chunk = ids[st: st + self.seq_len + 1].to(torch.long)
        return chunk[:-1], chunk[1:]

# --- Collect books and split each book's tokens ---
books = collect_books(rows)
if not books:
    raise RuntimeError('No tokenized books found. Check data/tokenized/*.ids.txt and metadata.csv token_count.')

train_chunks = []
val_chunks = []
test_chunks = []

total_train_tokens = 0
total_val_tokens = 0
total_test_tokens = 0

for b in books:
    ids = load_ids_file(b.ids_path)
    tr, va, te = split_book_tokens(ids, TRAIN_RATIO, VAL_RATIO)
    train_chunks.append(torch.tensor(tr, dtype=torch.int32))
    val_chunks.append(torch.tensor(va, dtype=torch.int32))
    test_chunks.append(torch.tensor(te, dtype=torch.int32))
    total_train_tokens += len(tr)
    total_val_tokens += len(va)
    total_test_tokens += len(te)

train_ds = CachedWindowDataset(train_chunks, SEQ_LEN, STRIDE)
val_ds   = CachedWindowDataset(val_chunks, SEQ_LEN, STRIDE)
test_ds  = CachedWindowDataset(test_chunks, SEQ_LEN, STRIDE)

print(f'Books: {len(books)} (each split {TRAIN_RATIO:.0%}/{VAL_RATIO:.0%}/{TEST_RATIO:.0%})')
print(f'Tokens: train={total_train_tokens:,}  val={total_val_tokens:,}  test={total_test_tokens:,}')
print(f'Samples: train={len(train_ds):,}  val={len(val_ds):,}  test={len(test_ds):,}')

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True,
    drop_last=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

Books: 49 (each split 80%/10%/10%)
Tokens: train=2,592,401  val=324,048  test=324,076
Samples: train=259,019  val=32,185  test=32,187


## Model

In [4]:
class LSTMLM(nn.Module):
    def __init__(self, vocab_size: int, emb_dim: int, hidden: int, num_layers: int, dropout: float, pad_id: int):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_id)
        self.lstm = nn.LSTM(
            input_size=emb_dim,
            hidden_size=hidden,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden, vocab_size)

    def forward(self, x):
        e = self.emb(x)          # (B,T,E)
        out, _ = self.lstm(e)    # (B,T,H)
        out = self.drop(out)
        logits = self.fc(out)    # (B,T,V)
        return logits

## Sampling utilities

In [5]:
@torch.no_grad()
def sample_ids(model: nn.Module, prompt_ids: List[int], max_new_tokens: int = 120,
               temperature: float = 1.0, top_k: int = 50) -> List[int]:
    model.eval()
    x = torch.tensor([prompt_ids], dtype=torch.long, device=DEVICE)
    for _ in range(max_new_tokens):
        logits = model(x)[:, -1, :]  # (1,V)
        logits = logits / max(temperature, 1e-6)

        if top_k and top_k > 0:
            v, ix = torch.topk(logits, k=min(top_k, logits.size(-1)))
            probs = torch.softmax(v, dim=-1)
            next_local = torch.multinomial(probs, num_samples=1)
            next_id = ix.gather(-1, next_local)
        else:
            probs = torch.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)

        next_id_int = int(next_id.item())
        x = torch.cat([x, torch.tensor([[next_id_int]], device=DEVICE)], dim=1)

        if eos_id is not None and next_id_int == eos_id:
            break
        if x.size(1) > 512:
            x = x[:, -512:]

    return x[0].tolist()

def decode(ids: List[int]) -> str:
    toks = [vocab[i] if 0 <= i < len(vocab) else '<unk>' for i in ids]
    text = ' '.join(toks)
    text = (text
            .replace(' ,', ',').replace(' .', '.').replace(' !', '!').replace(' ?', '?')
            .replace(' ;', ';').replace(' :', ':')
            .replace(' )', ')').replace('( ', '(')
            .replace(' - ', '-')
           )
    return text

## Train + evaluate

In [6]:
criterion = nn.CrossEntropyLoss(ignore_index=pad_id)

def run_eval(model, loader=None) -> Tuple[float, float]:
    if loader is None:
        loader = val_loader
    model.eval()
    total_loss = 0.0
    total_tokens = 0

    with torch.no_grad():
        for x, y in loader:
            x = x.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)

            with autocast("cuda", enabled=use_amp):
                logits = model(x)
                loss = criterion(logits.reshape(-1, logits.size(-1)), y.reshape(-1))

            mask = (y != pad_id)
            tok = int(mask.sum().item())
            total_tokens += tok
            total_loss += float(loss.item()) * tok

    avg_loss = total_loss / max(1, total_tokens)
    ppl = math.exp(min(20.0, avg_loss))
    return avg_loss, ppl

## Tests

In [7]:
def encode_prompt(text: str) -> List[int]:
    toks = text.split()
    ids = [tok2id.get(t, tok2id.get('<unk>', 0)) for t in toks]
    return ids

In [8]:
import itertools

# ---------- Search space ----------
SEARCH_GRID = {
    'lr':          [0.0005],
    'hidden_size': [4096],
    'dropout':     [0.6],
}

SEARCH_EMB_DIM    = 512
SEARCH_NUM_LAYERS = 2
SEARCH_EPOCHS     = 25
SEARCH_STEPS_PER_EPOCH = 1500
SEARCH_BATCH_SIZE = 256
SEARCH_SEQ_LEN    = 50
SEARCH_STRIDE     = 10
EARLY_STOP_PATIENCE = 2        # stop after 3 epochs with no val improvement

# ReduceLROnPlateau settings
SCHED_FACTOR  = 0.5            # halve LR when plateau
SCHED_PATIENCE = 1             # reduce after 1 epoch of no improvement
SCHED_MIN_LR  = 1e-5           # don't go below this

# Build all configs
keys = list(SEARCH_GRID.keys())
combos = list(itertools.product(*[SEARCH_GRID[k] for k in keys]))
configs = [dict(zip(keys, c)) for c in combos]

print(f"Total configurations: {len(configs)}")
for i, cfg in enumerate(configs):
    print(f"  [{i+1}] lr={cfg['lr']}, hidden={cfg['hidden_size']}, dropout={cfg['dropout']}")

Total configurations: 1
  [1] lr=0.0005, hidden=4096, dropout=0.6


In [9]:
def run_search_eval(mdl, loader=None) -> Tuple[float, float]:
    """Evaluate model on a given loader, return (avg_loss, perplexity)."""
    if loader is None:
        loader = val_loader
    mdl.eval()
    total_loss = 0.0
    total_tokens = 0
    with torch.no_grad():
        for x, y in loader:
            x = x.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)
            with autocast("cuda", enabled=use_amp):
                logits = mdl(x)
                loss = criterion(logits.reshape(-1, logits.size(-1)), y.reshape(-1))
            mask = (y != pad_id)
            tok = int(mask.sum().item())
            total_tokens += tok
            total_loss += float(loss.item()) * tok
    avg_loss = total_loss / max(1, total_tokens)
    ppl = math.exp(min(20.0, avg_loss))
    return avg_loss, ppl


search_results = []
best_model_state = None  # keep best model weights for test eval

actual_steps = min(SEARCH_STEPS_PER_EPOCH, len(train_loader))

for cfg_idx, cfg in enumerate(configs):
    print(f"\n{'='*60}")
    print(f"Config [{cfg_idx+1}/{len(configs)}]  lr={cfg['lr']}  hidden={cfg['hidden_size']}  dropout={cfg['dropout']}")
    print(f"Steps per epoch: {actual_steps} (loader has {len(train_loader)} batches, cap {SEARCH_STEPS_PER_EPOCH})")
    print('='*60)

    set_seed(SEED)

    mdl = LSTMLM(
        vocab_size=len(vocab),
        emb_dim=SEARCH_EMB_DIM,
        hidden=cfg['hidden_size'],
        num_layers=SEARCH_NUM_LAYERS,
        dropout=cfg['dropout'],
        pad_id=pad_id,
    ).to(DEVICE)

    crit = nn.CrossEntropyLoss(ignore_index=pad_id)
    opt  = torch.optim.Adam(mdl.parameters(), lr=cfg['lr'])
    scl  = GradScaler("cuda", enabled=use_amp)

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt, mode='min', factor=SCHED_FACTOR,
        patience=SCHED_PATIENCE, min_lr=SCHED_MIN_LR
    )

    best_val_loss = float('inf')
    patience_counter = 0
    best_epoch = 0
    best_state = None

    for epoch in range(1, SEARCH_EPOCHS + 1):
        mdl.train()
        running = 0.0
        seen_tokens = 0

        pbar = tqdm(enumerate(train_loader, start=1),
                     total=actual_steps,
                     desc=f"  E{epoch}/{SEARCH_EPOCHS}", leave=False)

        for step, (x, y) in pbar:
            x = x.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)

            opt.zero_grad(set_to_none=True)
            with autocast("cuda", enabled=use_amp):
                logits = mdl(x)
                loss = crit(logits.reshape(-1, logits.size(-1)), y.reshape(-1))

            scl.scale(loss).backward()
            scl.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(mdl.parameters(), GRAD_CLIP)
            scl.step(opt)
            scl.update()

            mask = (y != pad_id)
            tok = int(mask.sum().item())
            running += float(loss.item()) * tok
            seen_tokens += tok

            avg = running / max(1, seen_tokens)
            pbar.set_postfix(loss=f"{avg:.4f}")

            if step >= actual_steps:
                break
        pbar.close()

        val_loss, val_ppl = run_search_eval(mdl)
        train_loss = running / max(1, seen_tokens)
        current_lr = opt.param_groups[0]['lr']
        print(f"  epoch {epoch}  train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  val_ppl={val_ppl:.2f}  lr={current_lr:.6f}", end="")

        scheduler.step(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_epoch = epoch
            patience_counter = 0
            best_state = {k: v.cpu().clone() for k, v in mdl.state_dict().items()}
            print("  *best*")
        else:
            patience_counter += 1
            print(f"  (no improve {patience_counter}/{EARLY_STOP_PATIENCE})")
            if patience_counter >= EARLY_STOP_PATIENCE:
                print(f"  >> early stop at epoch {epoch}, best was epoch {best_epoch}")
                break

    search_results.append({
        **cfg,
        'best_val_loss': best_val_loss,
        'best_val_ppl': math.exp(min(20.0, best_val_loss)),
        'best_epoch': best_epoch,
        'total_epochs': epoch,
        'best_state': best_state,
    })

    # free GPU memory before next config
    del mdl, opt, scl, crit, scheduler
    torch.cuda.empty_cache()

print("\n\nGrid search complete.")


Config [1/1]  lr=0.0005  hidden=4096  dropout=0.6
Steps per epoch: 1011 (loader has 1011 batches, cap 1500)


  E1/25:   0%|          | 0/1011 [00:00<?, ?it/s]

  epoch 1  train_loss=4.6085  val_loss=4.5608  val_ppl=95.66  lr=0.000500  *best*


  E2/25:   0%|          | 0/1011 [00:00<?, ?it/s]

  epoch 2  train_loss=3.5574  val_loss=4.6387  val_ppl=103.41  lr=0.000500  (no improve 1/2)


  E3/25:   0%|          | 0/1011 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# Sort by best validation loss and display results
search_results.sort(key=lambda r: r['best_val_loss'])

print(f"{'Rank':<5} {'LR':<10} {'Hidden':<8} {'Dropout':<9} {'Val Loss':<10} {'Val PPL':<10} {'Best@':<7} {'Stopped':<7}")
print('-' * 70)
for i, r in enumerate(search_results):
    print(f"{i+1:<5} {r['lr']:<10} {r['hidden_size']:<8} {r['dropout']:<9.1f} {r['best_val_loss']:<10.4f} {r['best_val_ppl']:<10.2f} {r['best_epoch']:<7} {r['total_epochs']:<7}")

best = search_results[0]
print(f"\nBest config:")
print(f"  lr={best['lr']}, hidden_size={best['hidden_size']}, dropout={best['dropout']}")
print(f"  val_loss={best['best_val_loss']:.4f}, val_ppl={best['best_val_ppl']:.2f}, best at epoch {best['best_epoch']}")

# --- Final test evaluation (only done once, on best model) ---
print(f"\n{'='*60}")
print("Final TEST evaluation (best model from grid search)")
print('='*60)

best_mdl = LSTMLM(
    vocab_size=len(vocab),
    emb_dim=SEARCH_EMB_DIM,
    hidden=best['hidden_size'],
    num_layers=SEARCH_NUM_LAYERS,
    dropout=best['dropout'],
    pad_id=pad_id,
).to(DEVICE)
best_mdl.load_state_dict(best['best_state'])

test_loss, test_ppl = run_search_eval(best_mdl, loader=test_loader)
print(f"  test_loss={test_loss:.4f}  test_ppl={test_ppl:.2f}")

# Clean up saved states from search_results to free memory
for r in search_results:
    r.pop('best_state', None)
del best_mdl
torch.cuda.empty_cache()